Academic Integrity and Learning Statement

By submitting my work, I confirm that:

1. The code, analysis, and documentation in this notebook are my own work and reflect my own understanding.
2. I am prepared to explain all code and analysis included in this submission.

If I used assistance (e.g., AI tools, tutors, or other resources), I have:

- Clearly documented where and how external tools or resources were used in my solution.
- Included a copy of the interaction (e.g., AI conversation or tutoring notes) in an appendix.

I acknowledge that:

- I may be asked to explain any part of my code or analysis during evaluation.
- Misrepresenting assisted work as my own constitutes academic dishonesty and undermines my learning.

In [ ]:
# Import core python libraries
import numpy as np
import os, platform, multiprocessing, subprocess
import pandas as pd

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Import custom python files
from src.utils import logging as prjlogger
from src.config import settings as prjconfigs
from src.visualisation import plots as prjplots
from src.data import loader as dloader
from src.features import engineering as prjfengg

In [ ]:
# Enable auto-reload extension
%load_ext autoreload
# Automatically reload all modules before executing code
%autoreload 2
%reload_ext autoreload

In [ ]:
# Check software specs
dict_sw_version = {
    'python': os.popen('python --version').read().strip(),
    'host platform': platform.platform(),
    'numpy': np.__version__,
    'pandas': pd.__version__
}

for key, value in dict_sw_version.items():
    print(f'{prjplots.beautify(key, 1)} version is: {prjplots.beautify(value)}')

In [ ]:
logger = prjlogger.get_logger()
logger.debug(f"Initialised logging for {prjconfigs.PROJECT_NAME} project.")

In [ ]:
# Check CPU cores
print(f'CPU cores available to use: {prjplots.beautify(str(multiprocessing.cpu_count()))}')

# Machine info
print(f"Processor: {prjplots.beautify(platform.processor())}")
print(f"Machine: {prjplots.beautify(platform.machine())}")

# GPU info
print("PyTorch MPS (Metal) Status:")
print(f"MPS available: {prjplots.beautify(torch.backends.mps.is_available())}")
print(f"MPS built: {prjplots.beautify(str(torch.backends.mps.is_built()))}")
print(f"CUDA available: {prjplots.beautify(str(torch.cuda.is_available()))}")

In [ ]:
df_raw_train = dloader.load_data(prjconfigs.TRAIN_FILE, True)
df_raw_test = dloader.load_data(prjconfigs.TEST_FILE, True)

In [ ]:
df_raw_train.sample(3)

In [ ]:
# Columns tagging
insignificant_cols = ['Order', 'PID']
target_col = 'SalePrice'
ignorable_cols = insignificant_cols + [target_col]

temporal_cols_name_pattern = ['Yr', 'Year']

ordinal_cols = ['LotShape', 'Utilities', 'LandSlope', 'OverallQual', 'OverallCond', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

In [ ]:
# Merge train and test data
df_raw_all, df_raw_target = dloader.merge_train_test_data(
    df_raw_train,
    df_raw_test,
    insignificant_cols,
    target_col
)

In [ ]:
# Split into train and test based on the is_train column
df_train = df_raw_all[df_raw_all['is_train'] == 1].iloc[:, :-1]
df_test = df_raw_all[df_raw_all['is_train'] == 0].iloc[:, :-1]

In [ ]:
# Classify features
feature_categories = prjfengg.classify_columns(
    df=df_train,
    n_cat_threshold=prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_ABS_VAL,
    threshold_type=prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_TYPE_ABS,
    cols_to_ignore=ignorable_cols,
    temporal_cols_name_pattern=temporal_cols_name_pattern,
    ordinal_cols=ordinal_cols
)

In [ ]:
# Fetch column categories
(cols_num_continuous, n_cols_num_continuous, cols_num_discrete, n_cols_num_discrete,
 cols_cat_nominal, n_cols_cat_nominal, cols_cat_ordinal, n_cols_cat_ordinal,
 cols_object, n_cols_object, cols_temporal, n_cols_temporal,
 cols_binary, n_cols_binary, cols_low_cardinality, n_cols_low_cardinality) = prjfengg.get_cols_as_tuple(feature_categories)

In [ ]:
n_total_col_before_feat_split = df_train.shape[1]
n_total_col_after_feat_split = n_cols_num_continuous + n_cols_num_discrete + n_cols_cat_nominal + n_cols_cat_ordinal + n_cols_object + n_cols_temporal + n_cols_binary

print(f"="*72)
print(
    f"Total raw columns      = {prjplots.beautify(str(len(df_train.columns)))}\n"
    f"Numerical Continuous   = {prjplots.beautify(n_cols_num_continuous)}\n"
    f"Numerical Discrete     = {prjplots.beautify(n_cols_num_discrete)}\n"
    f"Categorical Nominal    = {prjplots.beautify(n_cols_cat_nominal)}\n"
    f"Categorical Ordinal    = {prjplots.beautify(n_cols_cat_ordinal)}\n"
    f"Object/String          = {prjplots.beautify(n_cols_object)}\n"
    f"Temporal               = {prjplots.beautify(n_cols_temporal)}\n"
    f"Binary                 = {prjplots.beautify(n_cols_binary)}\n"
    f"Total feature columns  = {prjplots.beautify(n_total_col_after_feat_split)}"
)

has_inconsistency = n_total_col_before_feat_split != n_total_col_after_feat_split
inconsistency_val = prjplots.beautify("True", 3) if has_inconsistency else prjplots.beautify("False", 1)
print("=" * 72)
print(f"Any inconsistencies detected?[True/False] = {inconsistency_val}")
print(f'='*72)

## Task 1: EDA and Data cleaning

In [ ]:
# Calculate the number of NaN values for each column
nan_counts = df_train.isna().sum()

# Filter only columns that have NaN values and sort by the number of NaNs
cols_with_nans = nan_counts[nan_counts > 0].index.tolist()
print(f"Columns with NaNs: = {prjplots.beautify(str(len(cols_with_nans)))}/{prjplots.beautify(n_total_col_before_feat_split)}")
print(f"And they are: {cols_with_nans}")

In [ ]:
# Create a temp DF to capture column cardinality info
df_cardinality = prjfengg.get_cardinality_df(df_train)
df_cardinality.sample(3)

### Task 1.1: Exploring missing data

In [ ]:
# Plot cardinality distribution of features in the dataset
prjplots.plot_cardinality(df_cardinality, prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_ABS_VAL, threshold_used=prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_TYPE_ABS, type_of_cols='all', figsize=(20, 6))

In [ ]:
# Dive into cols having NaNs only
prjplots.plot_cardinality(df_cardinality[df_cardinality['col_name'].isin(cols_with_nans)], prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_ABS_VAL, threshold_used=prjconfigs.CATEGORICAL_CARDINALITY_THRESHOLD_TYPE_ABS, type_of_cols="NaN", figsize=(10, 6))

In [ ]:
# Creating a copy of the raw data to impute missing values for plotting purposes only (as NaNs are not plotted)
df_imputed_for_plots = df_train.copy()
df_imputed_for_plots[cols_num_continuous] = df_imputed_for_plots[cols_num_continuous].fillna(0)
most_frequent = df_imputed_for_plots[cols_num_discrete].mode().iloc[0]
df_imputed_for_plots[cols_num_discrete] = df_imputed_for_plots[cols_num_discrete].fillna(most_frequent)

In [ ]:
# Plot correlation between numerical features and target variable
correlation_plot = prjplots.plot_correlation_with_target(pd.concat([df_imputed_for_plots[cols_num_continuous], df_raw_target], axis=1), target_col)

In [ ]:
prjplots.plot_numerical_distribution(df_imputed_for_plots, cols_num_continuous)

In [ ]:
prjplots.plot_categorical_distribution(df_imputed_for_plots, cols_cat_nominal)

In [ ]:
prjplots.plot_categorical_distribution(df_imputed_for_plots, cols_cat_ordinal)

In [ ]:
prjplots.plot_relationship_to_target(pd.concat([df_imputed_for_plots[cols_num_discrete], df_raw_target], axis=1), cols_num_discrete, target_col)

In [ ]:
prjplots.plot_relationship_to_target(pd.concat([df_imputed_for_plots[cols_num_discrete], df_raw_target], axis=1), cols_num_discrete, target_col, trend_type='median')

In [ ]:
# Split train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    df_train,
    df_raw_target,
    test_size=prjconfigs.VALIDATION_SIZE,
    random_state=prjconfigs.RANDOM_STATE
)

In [ ]:
def pad(s, w=14): return prjplots.beautify(s.ljust(w))

rows = [
  ("Raw TRAIN data variable:",        "df_raw_train",  df_raw_train.shape),
  ("Raw TEST data variable:",         "df_raw_test",   df_raw_test.shape),
  ("Raw ALL data variable:",          "df_raw_all",    df_raw_all.shape),
  ("Raw TARGET data variable:",       "df_raw_target", df_raw_target.shape),
  ("TRAIN data variable:",            "df_train",      df_train.shape),
  ("TEST data variable:",             "df_test",       df_test.shape),
  ("Validation TRAIN data variable:", "X_train",       X_train.shape),
  ("Validation VAL data variable:",   "X_val",         X_val.shape),
]

for label, varname, shape in rows:
  print(f"{label:<32}{pad(varname)} having shape = {prjplots.beautify(shape, 1)}")

In [ ]:
num_columns = cols_num_continuous
cat_columns = cols_cat_nominal + cols_num_discrete + cols_binary + cols_object # included object as it is low cardinality
cat_ordinal_columns = cols_cat_ordinal
tempo_columns = cols_temporal

In [ ]:
len(num_columns), len(cat_columns), len(cat_ordinal_columns), len(tempo_columns)

In [ ]:
# Probing the unique values of each categorical column to be used for ordinal categorisation later
for col in cols_cat_ordinal:
  unique_vals = sorted(df_train[col].dropna().unique().tolist())
  print(f"{col:<15} {unique_vals}")

In [ ]:
# Initialising the preprocessing pipeline
pproc_pipe = prjfengg.create_pproc_pipeline(num_columns, cat_columns, cat_ordinal_columns, tempo_columns)

In [ ]:
# Display an interactive visual summary of the preprocessing pipeline
pproc_pipe

In [ ]:
# Display columns passed through unprocessed
assigned = set(num_columns + cat_columns + cat_ordinal_columns + tempo_columns)
remainder_cols = [c for c in X_train.columns if c not in assigned]
print(f"Remainder cols ({len(remainder_cols)}): {remainder_cols}")

In [ ]:
# Print feature expansion mapping for each transformer branch
prjfengg.print_feature_expansion(pproc_pipe)

_fit_ learns the transformation parameters from the data — for example, the imputer learns the median/most-frequent value of each column, the scaler learns the mean and std, the ordinal encoder learns the category mappings. These learned values are stored inside the transformer objects.

_transform_ then applies those learned parameters to any data you pass in — it doesn't learn anything new, just applies what was learned during fit.

In [ ]:
# Fit the preprocessing pipeline on training data and transform it
# fit only on training data — so the pipeline learns statistics
# (mean, std, most frequent values, etc.) exclusively from training data
X_train_transformed = pproc_pipe.fit_transform(X_train)

In [ ]:
# Transform validation data using the already-fitted pipeline
# transform validation/test data using those same training-derived statistics —
# this prevents data leakage, where information from val/test data
# influences how the training data is preprocessed
X_val_transformed = pproc_pipe.transform(X_val)

In [ ]:
y_train_transformed = y_train.to_numpy()
y_val_transformed = y_val.to_numpy()

In [ ]:
# Check for both NaN and None
has_nulls_or_nans = pd.isna(X_train_transformed).any()
print(f"Contains null or NaN values: {has_nulls_or_nans}")